# Model Training and Evaluation
This notebook trains the EfficientNet-B0 model, evaluates its performance, and presents results with analysis and visualization.

In [1]:
# 1. Import Libraries
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [4]:
# 2. Load Model and Data
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

# Load model (if saved previously, else use model from previous notebook)
# model = load_model('efficientnetb0_model.h5')

labels = pd.read_csv('labels_preprocessed.csv')
img_size = 224
batch_size = 32

# Data generators (same as before)
datagen = ImageDataGenerator(rescale=1./255)

df_train, df_val = train_test_split(labels, test_size=0.2, stratify=labels['breed_encoded'], random_state=42)

train_gen = datagen.flow_from_dataframe(
    df_train, directory='train/', x_col='img', y_col='breed',
    target_size=(img_size, img_size), batch_size=batch_size, class_mode='categorical')

val_gen = datagen.flow_from_dataframe(
    df_val, directory='train/', x_col='img', y_col='breed',
    target_size=(img_size, img_size), batch_size=batch_size, class_mode='categorical')

Found 8177 validated image filenames belonging to 120 classes.
Found 2045 validated image filenames belonging to 120 classes.


In [ ]:
# 3. Train the Model
# Use callbacks for best model saving and early stopping
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import os

checkpoint = ModelCheckpoint('efficientnetb0_best.h5', monitor='val_accuracy', save_best_only=True, mode='max')
earlystop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

# 3. Train the Model

checkpoint = ModelCheckpoint('efficientnetb0_best.h5', monitor='val_accuracy', save_best_only=True, mode='max')
earlystop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

if 'model' not in globals():
    if os.path.exists('efficientnetb0_best.h5'):
        model = load_model('efficientnetb0_best.h5')
    else:
        num_classes = df_train['breed_encoded'].nunique()
        base = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(img_size, img_size, 3))
        x = base.output
        x = tf.keras.layers.GlobalAveragePooling2D()(x)
        x = tf.keras.layers.Dropout(0.2)(x)
        outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
        model = tf.keras.models.Model(inputs=base.input, outputs=outputs)
        base.trainable = False
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=[checkpoint, earlystop]
)

NameError: name 'model' is not defined

In [ ]:
# 4. Evaluate Model Performance
# Predict on validation set
y_true = df_val['label_encoded'].values
y_pred = model.predict(val_gen)
y_pred_classes = np.argmax(y_pred, axis=1)

print('Classification Report:')
print(classification_report(y_true, y_pred_classes))

print('Confusion Matrix:')
print(confusion_matrix(y_true, y_pred_classes))

In [ ]:
# 5. Visualize Training Results
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.show()

# 6. Analyze and Present Results
- The model's performance is evaluated using accuracy, precision, recall, and F1-score.
- Training and validation curves help diagnose overfitting or underfitting.
- Confusion matrix and classification report provide detailed insights into class-wise performance.

**Conclusion:**
- Summarize findings, discuss potential improvements (e.g., more data, fine-tuning, augmentation), and next steps for deployment or further research.